In [1]:

!pip install folium openpyxl


In [3]:
import pandas as pd
import folium
from google.colab import files
from IPython.display import display

In [4]:
archivo = files.upload()

nombre_archivo = list(archivo.keys())[0]

df = pd.read_excel(nombre_archivo)

print("Primeras filas de la base de datos:")
display(df.head())

print("Columnas:")
print(df.columns.tolist())

Saving f1_2016.xlsx to f1_2016.xlsx
Primeras filas de la base de datos:


,Piloto,AUS,BHR,CHN,RUS,ESP,MCO,CAN,EUR,AUT,...,DEU,BEL,ITA,SGP,MYS,JPN,USA,MEX,BRA,ABD
0,N. ROSBERG,25,25,25,25,0,6,10,25,12,...,12,25,25,25,15,25,18,18,18,18
1,L. HAMILTON,18,15,6,18,0,25,25,10,25,...,25,15,18,15,0,15,25,25,25,25
2,D. RICCIARDO,12,12,12,0,12,18,6,6,10,...,18,18,10,18,25,8,15,15,4,10
3,S. VETTEL,15,0,18,0,15,12,18,18,0,...,10,8,15,10,0,12,12,10,10,15
4,M. VERSTAPPEN,1,8,4,0,25,0,12,4,18,...,15,0,6,8,18,18,0,12,15,12


Columnas:
['Piloto', 'AUS', 'BHR', 'CHN', 'RUS', 'ESP', 'MCO', 'CAN', 'EUR', 'AUT', 'GBR', 'HUN', 'DEU', 'BEL', 'ITA', 'SGP', 'MYS', 'JPN', 'USA', 'MEX', 'BRA', 'ABD']


In [5]:
# Limpiamos los nombres de las columnas
df.columns = df.columns.astype(str).str.strip()

# Eliminamos espacios adicionales de los nombres de los pilotos
df["Piloto"] = df["Piloto"].astype(str).str.strip()

# Seleccionamos los dos pilotos
pilotos_seleccionados = ["N. ROSBERG", "F. MASSA"]

# Filtramos los datos
datos_pilotos = df[
    df["Piloto"].isin(pilotos_seleccionados)
].copy()

display(datos_pilotos)

,Piloto,AUS,BHR,CHN,RUS,ESP,MCO,CAN,EUR,AUT,...,DEU,BEL,ITA,SGP,MYS,JPN,USA,MEX,BRA,ABD
0,N. ROSBERG,25,25,25,25,0,6,10,25,12,...,12,25,25,25,15,25,18,18,18,18
10,F. MASSA,10,4,8,10,4,1,0,1,0,...,0,1,2,0,0,2,6,2,0,2


In [7]:
# Seleccionamos las columnas numéricas de las carreras
columnas_carreras = datos_pilotos.select_dtypes(
    include="number"
).columns

# Calculamos la suma de puntos por piloto
datos_pilotos["Puntaje_Total"] = datos_pilotos[
    columnas_carreras
].sum(axis=1)

print("Puntaje total de cada piloto:")

display(
    datos_pilotos[["Piloto", "Puntaje_Total"]]
)

Puntaje total de cada piloto:


,Piloto,Puntaje_Total
0,N. ROSBERG,770
10,F. MASSA,106


In [8]:

pilotos_info = {
    "N. ROSBERG": {
        "nombre": "Nico Rosberg",
        "escuderia": "Mercedes",
        "fotografia": "https://commons.wikimedia.org/wiki/Special:FilePath/Nico_Rosberg_2016_Malaysia_1.jpg",
        "ciudad": "Wiesbaden, Alemania",
        "latitud": 50.0826,
        "longitud": 8.2415
    },

    "F. MASSA": {
        "nombre": "Felipe Massa",
        "escuderia": "Williams",
        "fotografia": "https://commons.wikimedia.org/wiki/Special:FilePath/Felipe_Massa_2016_Malaysia.jpg",
        "ciudad": "São Paulo, Brasil",
        "latitud": -23.5505,
        "longitud": -46.6333
    }
}

print(pilotos_info)

{'N. ROSBERG': {'nombre': 'Nico Rosberg', 'escuderia': 'Mercedes', 'fotografia': 'https://commons.wikimedia.org/wiki/Special:FilePath/Nico_Rosberg_2016_Malaysia_1.jpg', 'ciudad': 'Wiesbaden, Alemania', 'latitud': 50.0826, 'longitud': 8.2415}, 'F. MASSA': {'nombre': 'Felipe Massa', 'escuderia': 'Williams', 'fotografia': 'https://commons.wikimedia.org/wiki/Special:FilePath/Felipe_Massa_2016_Malaysia.jpg', 'ciudad': 'São Paulo, Brasil', 'latitud': -23.5505, 'longitud': -46.6333}}


In [18]:
# Creamos nuevamente el mapa sin utilizar OpenStreetMap
mapa = folium.Map(
    location=[15, -20],
    zoom_start=2,
    tiles=None
)

# Agregamos un mapa base de Esri
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Street Map",
    overlay=False,
    control=True
).add_to(mapa)

# Agregamos los marcadores de los pilotos
for piloto in pilotos_seleccionados:
    info = pilotos_info[piloto]

    puntaje = datos_pilotos.loc[
        datos_pilotos["Piloto"] == piloto,
        "Puntaje_Total"
    ].iloc[0]

    contenido = f"""
    <div style="width:250px; text-align:center;">
        <h2>{info['nombre']}</h2>
        <img src="{info['fotografia']}" width="220">
        <p><b>Escudería:</b> {info['escuderia']}</p>
        <p><b>Lugar de nacimiento:</b> {info['ciudad']}</p>
        <p><b>Puntaje total:</b> {puntaje} puntos</p>
    </div>
    """

    color = "blue" if piloto == "N. ROSBERG" else "red"

    folium.Marker(
        location=[info["latitud"], info["longitud"]],
        popup=folium.Popup(contenido, max_width=300),
        tooltip=info["nombre"],
        icon=folium.Icon(
            color=color,
            icon="flag-checkered",
            prefix="fa"
        )
    ).add_to(mapa)


In [13]:

# Recorremos los dos pilotos seleccionados
for piloto in pilotos_seleccionados:

    # Obtenemos información del piloto
    info = pilotos_info[piloto]

    # Obtenemos su puntaje total
    puntaje = datos_pilotos.loc[
        datos_pilotos["Piloto"] == piloto,
        "Puntaje_Total"
    ].iloc[0]

    # Creamos el contenido HTML del popup
    contenido = f"""
    <div style="width:250px; text-align:center;">

        <h2>{info['nombre']}</h2>

        <img src="{info['fotografia']}"
             width="220"
             style="border-radius:10px;">

        <p><b>Escudería:</b> {info['escuderia']}</p>

        <p><b>Lugar de nacimiento:</b>
        {info['ciudad']}</p>

        <p><b>Puntaje total:</b>
        {puntaje} puntos</p>

    </div>
    """

    # Creamos la ventana emergente
    popup = folium.Popup(
        contenido,
        max_width=300
    )

    # Personalizamos el color de los marcadores
    color = "blue" if piloto == "N. ROSBERG" else "red"

    # Agregamos marcador al mapa
    folium.Marker(
        location=[
            info["latitud"],
            info["longitud"]
        ],
        popup=popup,
        tooltip=info["nombre"],
        icon=folium.Icon(
            color=color,
            icon="flag-checkered",
            prefix="fa"
        )
    ).add_to(mapa)


In [19]:
# Ajustamos el mapa para visualizar ambos marcadores
coordenadas = [
    [info["latitud"], info["longitud"]]
    for info in pilotos_info.values()
]

mapa.fit_bounds(coordenadas)

# Mostramos el mapa interactivo
display(mapa)


Justificación del diseño del mapa y las ventanas emergentes

Para la elaboración del mapa interactivo se utilizó la biblioteca Folium, debido a que permite representar información geográfica mediante marcadores y ventanas emergentes. Se eligió OpenStreetMap como mapa base porque facilita la ubicación de las ciudades de nacimiento de los pilotos y permite visualizar sus posiciones geográficas.

Respecto a los marcadores, se utilizaron colores diferentes para distinguir a Nico Rosberg y Felipe Massa, lo que facilita identificar a cada piloto en el mapa. Asimismo, se incorporó un ícono relacionado con las carreras de automovilismo para mantener una relación visual con el tema de la Fórmula 1.

En las ventanas emergentes se emplearon etiquetas HTML como <h2> para destacar el nombre de cada piloto, <img> para incorporar su fotografía y <p>  para organizar la información de su escudería, lugar de nacimiento y puntaje total. También se utilizó <b> para resaltar los datos más importantes. Finalmente, se optó por una distribución centrada y ordenada para facilitar la lectura y permitir que el usuario identifique rápidamente la información de cada piloto. De esta manera, el mapa no solo presenta la ubicación geográfica, sino que también integra información deportiva y personal de los participantes de la temporada 2016.